# Promptable segmentation with SAM 3

Segment every object matching a concept (a text noun phrase or a bounding box) using Meta's [Segment Anything Model 3](https://huggingface.co/facebook/sam3).

**What's in this recipe:**

- Run text-prompted segmentation ("segment every cat")
- Run box-prompted segmentation (segment a specific region)
- Combine text and negative box prompts to refine results
- Visualize instance masks on top of the original image

## Problem

Classical segmentation models output a fixed set of class labels. SAM 3 instead performs *promptable concept segmentation*: you describe what you want ("yellow school bus", "handle", or a bounding box around the region of interest), and the model returns instance masks for every matching object.

| Approach | Output | Prompting |
|----------|--------|-----------|
| Panoptic segmentation (DETR) | Fixed COCO classes | None |
| Promptable segmentation (SAM 3) | Per-instance binary masks | Text and/or boxes |

## Solution

Use the `sam_for_segmentation` UDF in `pixeltable.functions.huggingface` to attach SAM 3 to an image column as a computed column.

### Setup

> **Note:** `facebook/sam3` is a gated repository. Request access on its [model page](https://huggingface.co/facebook/sam3), then authenticate with `huggingface-cli login` (or set the `HF_TOKEN` environment variable) before running this notebook.

In [ ]:
%pip install -qU pixeltable torch transformers

### Load images

In [ ]:
import numpy as np
import pixeltable as pxt
from pixeltable.functions.huggingface import sam_for_segmentation

pxt.drop_dir('sam3_demo', force=True)
pxt.create_dir('sam3_demo')

images = pxt.create_table('sam3_demo/images', {'image': pxt.Image})

base_url = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images'
images.insert(
    [
        {'image': f'{base_url}/000000000009.jpg'},
        {'image': f'{base_url}/000000000034.jpg'},
    ]
)

### Segment every instance matching a text concept

Pass a short noun phrase as the `text` argument. SAM 3 will return one binary mask per detected instance.

**Parameters:**

- `model_id`: defaults to `facebook/sam3`
- `text`: a short concept prompt (e.g., `'orange'`, `'yellow school bus'`)
- `threshold`: confidence threshold for filtering instances
- `mask_threshold`: threshold applied to the mask logits to produce binary masks

**Output:**

```python
{
    'scores': [0.92, 0.88],
    'boxes': [[x1, y1, x2, y2], ...],   # pixel coordinates
    'masks': np.ndarray,                # shape (num_instances, H, W), dtype=bool
}
```

In [ ]:
images.add_computed_column(
    oranges=sam_for_segmentation(
        images.image, text='orange', threshold=0.5
    )
)

In [ ]:
# Inspect the scores and boxes for each detected instance
images.select(
    images.image, scores=images.oranges.scores, boxes=images.oranges.boxes
).collect()

### Count and measure instances

Because the result is just a dict, you can derive scalar metrics with plain Python UDFs.

In [ ]:
@pxt.udf
def num_instances(result: dict) -> int:
    return len(result['scores'])


@pxt.udf
def total_mask_area(result: dict) -> int:
    masks = result['masks']
    return int(np.asarray(masks).sum())


images.select(
    images.image,
    n_oranges=num_instances(images.oranges),
    orange_pixels=total_mask_area(images.oranges),
).collect()

### Visualize masks on the original image

Combine the per-instance masks with their bounding boxes to render an overlay using a small UDF.

In [ ]:
import PIL.Image
import PIL.ImageDraw


@pxt.udf
def draw_sam_masks(
    image: PIL.Image.Image, result: dict
) -> PIL.Image.Image:
    base = image.convert('RGBA')
    overlay = PIL.Image.new('RGBA', base.size, (0, 0, 0, 0))
    draw = PIL.ImageDraw.Draw(overlay)
    masks = np.asarray(result['masks'])
    boxes = result['boxes']
    palette = [
        (255, 99, 71),
        (60, 179, 113),
        (65, 105, 225),
        (255, 215, 0),
        (138, 43, 226),
    ]
    for i, (mask, box) in enumerate(zip(masks, boxes)):
        color = palette[i % len(palette)]
        mask_img = PIL.Image.fromarray(
            (mask.astype(np.uint8) * 120), mode='L'
        )
        color_img = PIL.Image.new('RGBA', base.size, (*color, 0))
        color_img.putalpha(mask_img)
        overlay = PIL.Image.alpha_composite(overlay, color_img)
        draw = PIL.ImageDraw.Draw(overlay)
        draw.rectangle(box, outline=(*color, 255), width=3)
    return PIL.Image.alpha_composite(base, overlay).convert('RGB')


images.add_computed_column(
    orange_viz=draw_sam_masks(images.image, images.oranges)
)

images.select(images.image, images.orange_viz).collect()

### Segment by bounding box

Skip the text prompt and pass a bounding box (in `[x1, y1, x2, y2]` pixel coordinates) to segment a specific region. SAM 3 returns the matching concept's instance masks across the whole image.

In [ ]:
images.add_computed_column(
    box_seg=sam_for_segmentation(
        images.image, input_boxes=[[100, 150, 500, 450]]
    )
)

images.select(
    images.image, images.box_seg.scores, images.box_seg.boxes
).collect()

### Refine with negative box prompts

Combine a text prompt with one or more *negative* boxes (label `0`) to ask SAM 3 to ignore specific regions. This is useful when a concept matches multiple visually similar objects but you only want a subset.

In [ ]:
images.add_computed_column(
    handles_minus_oven=sam_for_segmentation(
        images.image,
        text='handle',
        input_boxes=[[40, 183, 318, 204]],
        input_boxes_labels=[0],
    )
)

images.select(
    images.image,
    scores=images.handles_minus_oven.scores,
    boxes=images.handles_minus_oven.boxes,
).collect()

## Explanation

SAM 3 differs from earlier segmentation models in two important ways:

1. **Concept-based prompting.** Instead of choosing a label from a fixed taxonomy, you describe the concept you want with free-form text, bounding boxes, or both.
1. **Instance masks for every match.** A single forward pass returns one mask per matching instance, with confidence scores and bounding boxes.

When stored as a Pixeltable computed column, every new row inserted into the table will be segmented automatically, and downstream computed columns (counts, areas, overlays) refresh on the same schedule.

## See also

- [Compare object detection and panoptic segmentation](./img-detection-vs-segmentation) - DETR-based detection and panoptic segmentation
- [Detect objects in images](./img-detect-objects) - Object detection with YOLOX
- [SAM 3 documentation](https://huggingface.co/docs/transformers/model_doc/sam3) - Hugging Face model docs